# Analyse des données depuis S3

Ce notebook sert d'outil d'exploration pour lire des fichiers stockés dans Amazon S3, les charger dans pandas, puis les préparer pour une analyse ou un traitement ultérieur.

In [35]:
import sys
import subprocess

# Installe les dépendances nécessaires dans l'environnement du notebook
subprocess.check_call([sys.executable, "-m", "pip", "install", "boto3", "pandas", "python-dotenv","pyproj "])

import boto3
import json
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

True

In [36]:
s3 = boto3.client("s3")

response = s3.get_object(
    Bucket="carburants-data-elodie",
    Key="2026-07-18_17-42/prix_bruts.json"
)

data = json.loads(response["Body"].read())

print("Nombre de stations :", len(data))
df = pd.DataFrame(data)
df.head()

Nombre de stations : 9799


,id,latitude,longitude,cp,pop,adresse,ville,horaires,services,prix,...,carburants_indisponibles,carburants_rupture_temporaire,carburants_rupture_definitive,horaires_automate_24_24,services_service,departement,code_departement,region,code_region,horaires_jour
0,25115001,4725741.3021458,592408.72885659,25115,R,Z.A. les Salines,Pouilley-les-Vignes,"{""@automate-24-24"": ""1"", ""jour"": [{""@id"": ""1"",...","{""service"": [""Toilettes publiques"", ""Laverie"",...","[{""@nom"": ""Gazole"", ""@id"": ""1"", ""@maj"": ""2026-...",...,[E10],NaN,E10,Oui,"[Toilettes publiques, Laverie, Relais colis, S...",Doubs,25,Bourgogne-Franche-Comté,27,Automate-24-24
1,56580001,4806100,-269000,56580,R,16 rue chateaubriand,Bréhan,"{""@automate-24-24"": ""1"", ""jour"": [{""@id"": ""1"",...","{""service"": [""Relais colis"", ""Station de gonfl...","[{""@nom"": ""Gazole"", ""@id"": ""1"", ""@maj"": ""2026-...",...,"[E85, GPLc, E10]",NaN,E85;GPLc;E10,Oui,"[Relais colis, Station de gonflage, Aire de ca...",Morbihan,56,Bretagne,53,Automate-24-24
2,92230008,4893270,230440,92230,R,192 Avenue Louis Roche,Gennevilliers,NaN,NaN,"[{""@nom"": ""Gazole"", ""@id"": ""1"", ""@maj"": ""2026-...",...,"[E85, GPLc]",NaN,E85;GPLc,Non,None,Hauts-de-Seine,92,Île-de-France,11,NaN
3,34150003,4365100,354700,34150,R,Route de Pézenas,Gignac,"{""@automate-24-24"": ""1"", ""jour"": [{""@id"": ""1"",...","{""service"": [""Boutique alimentaire"", ""Bar"", ""S...","[{""@nom"": ""Gazole"", ""@id"": ""1"", ""@maj"": ""2026-...",...,"[SP95, GPLc]",NaN,SP95;GPLc,Oui,"[Boutique alimentaire, Bar, Station de gonflag...",Hérault,34,Occitanie,76,"Automate-24-24, Lundi01.00-01.00, Mardi, Mercr..."
4,38230003,4574500,518300,38230,R,Rue des Ardennes,Tignieu-Jameyzieu,"{""@automate-24-24"": ""1"", ""jour"": [{""@id"": ""1"",...","{""service"": ""Automate CB 24/24""}","[{""@nom"": ""Gazole"", ""@id"": ""1"", ""@maj"": ""2026-...",...,None,NaN,NaN,Oui,[Automate CB 24/24],Isère,38,Auvergne-Rhône-Alpes,84,"Automate-24-24, Lundi01.00-01.00, Mardi01.00-0..."


In [37]:
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)

(9799, 47)
['id', 'latitude', 'longitude', 'cp', 'pop', 'adresse', 'ville', 'horaires', 'services', 'prix', 'rupture', 'geom', 'gazole_maj', 'gazole_prix', 'sp95_maj', 'sp95_prix', 'e85_maj', 'e85_prix', 'gplc_maj', 'gplc_prix', 'e10_maj', 'e10_prix', 'sp98_maj', 'sp98_prix', 'e10_rupture_debut', 'e10_rupture_type', 'sp98_rupture_debut', 'sp98_rupture_type', 'sp95_rupture_debut', 'sp95_rupture_type', 'e85_rupture_debut', 'e85_rupture_type', 'gplc_rupture_debut', 'gplc_rupture_type', 'gazole_rupture_debut', 'gazole_rupture_type', 'carburants_disponibles', 'carburants_indisponibles', 'carburants_rupture_temporaire', 'carburants_rupture_definitive', 'horaires_automate_24_24', 'services_service', 'departement', 'code_departement', 'region', 'code_region', 'horaires_jour']
id                                 int64
latitude                             str
longitude                            str
cp                                   str
pop                                  str
adresse         

In [38]:
# Stats sur les prix
cols_prix = ["gazole_prix", "sp95_prix", "sp98_prix", 
             "e10_prix", "e85_prix", "gplc_prix"]

print("📊 Statistiques des prix :")
print(df[cols_prix].describe())

print("\n❌ Valeurs manquantes :")
print(df[cols_prix].isnull().sum())

📊 Statistiques des prix :
       gazole_prix    sp95_prix   sp98_prix     e10_prix     e85_prix  \
count  9586.000000  3035.000000  7723.00000  7497.000000  3890.000000   
mean      2.093082     2.016993     2.08221     1.997378     0.845622   
std       0.085708     0.053617     0.08009     0.070800     0.075182   
min       1.204000     1.703000     1.82600     1.829000     0.714000   
25%       2.039000     1.989000     2.02900     1.954000     0.809000   
50%       2.089000     1.999000     2.06900     1.985000     0.816500   
75%       2.130000     2.044500     2.11300     2.022000     0.859000   
max       2.800000     2.509000     2.69900     2.559000     2.069000   

         gplc_prix  
count  1459.000000  
mean      1.047862  
std       0.090424  
min       0.809000  
25%       0.995000  
50%       1.049000  
75%       1.094000  
max       2.030000  

❌ Valeurs manquantes :
gazole_prix     213
sp95_prix      6764
sp98_prix      2076
e10_prix       2302
e85_prix       5909
gpl

In [39]:
# Cellule — Détection automatique par IQR
def detecter_anomalies_iqr(df, colonne):
    Q1 = df[colonne].quantile(0.25)
    Q3 = df[colonne].quantile(0.75)
    IQR = Q3 - Q1
    
    borne_basse = Q1 - 1.5 * IQR
    borne_haute = Q3 + 1.5 * IQR
    
    anomalies = df[
        (df[colonne] < borne_basse) | 
        (df[colonne] > borne_haute)
    ]
    
    print(f"📊 {colonne}")
    print(f"   Borne basse : {borne_basse:.3f}€")
    print(f"   Borne haute : {borne_haute:.3f}€")
    print(f"   Anomalies   : {len(anomalies)} stations")
    print()
    
    return borne_basse, borne_haute

# Appliquer sur tous les carburants
cols_prix = ["gazole_prix", "sp95_prix", "sp98_prix",
             "e10_prix", "e85_prix", "gplc_prix"]

bornes = {}
for col in cols_prix:
    if df[col].notna().sum() > 0:
        bornes[col] = detecter_anomalies_iqr(df, col)

📊 gazole_prix
   Borne basse : 1.903€
   Borne haute : 2.266€
   Anomalies   : 295 stations

📊 sp95_prix
   Borne basse : 1.906€
   Borne haute : 2.128€
   Anomalies   : 94 stations

📊 sp98_prix
   Borne basse : 1.903€
   Borne haute : 2.239€
   Anomalies   : 297 stations

📊 e10_prix
   Borne basse : 1.852€
   Borne haute : 2.124€
   Anomalies   : 282 stations

📊 e85_prix
   Borne basse : 0.734€
   Borne haute : 0.934€
   Anomalies   : 410 stations

📊 gplc_prix
   Borne basse : 0.847€
   Borne haute : 1.242€
   Anomalies   : 43 stations



In [22]:
from datetime import datetime


cols_utiles = [
    "id", "adresse", "ville", "cp",
    "departement", "region", "code_departement",
    "latitude", "longitude",
    "gazole_prix", "sp95_prix", "sp98_prix",
    "e10_prix", "e85_prix", "gplc_prix",
    "carburants_disponibles", "horaires_automate_24_24"
]

df_silver = df[cols_utiles].copy()

# Corriger les types
    
df_silver["cp"] = df_silver["cp"].astype(str).str.zfill(5)

# Conserver les valeurs extrêmes si elles reflètent des différences réelles
# de prix entre stations, carburants ou périodes
cols_prix = ["gazole_prix", "sp95_prix", "sp98_prix",
             "e10_prix", "e85_prix", "gplc_prix"]

for col in cols_prix:
    df_silver[col] = pd.to_numeric(df_silver[col], errors="coerce")

# Ajouter date de collecte
df_silver["date_collecte"] = datetime.now()

# Supprimer stations sans aucun prix
df_silver = df_silver.dropna(subset=cols_prix, how="all")

print(f"✅ Stations après nettoyage : {len(df_silver)}")
print(f"\n❌ Valeurs manquantes après nettoyage :")
print(df_silver[cols_prix].isnull().sum())
df_silver.head()

✅ Stations après nettoyage : 9669

❌ Valeurs manquantes après nettoyage :
gazole_prix      83
sp95_prix      6634
sp98_prix      1946
e10_prix       2172
e85_prix       5779
gplc_prix      8210
dtype: int64


,id,adresse,ville,cp,departement,region,code_departement,latitude,longitude,gazole_prix,sp95_prix,sp98_prix,e10_prix,e85_prix,gplc_prix,carburants_disponibles,horaires_automate_24_24,date_collecte
0,25115001,Z.A. les Salines,Pouilley-les-Vignes,25115,Doubs,Bourgogne-Franche-Comté,25,4725741.3021458,592408.72885659,2.099,2.080,2.110,NaN,0.799,0.999,"[Gazole, SP95, E85, GPLc, SP98]",Oui,2026-07-20 13:00:01.911887
1,56580001,16 rue chateaubriand,Bréhan,56580,Morbihan,Bretagne,56,4806100,-269000,2.149,2.099,2.129,NaN,NaN,NaN,"[Gazole, SP95, SP98]",Oui,2026-07-20 13:00:01.911887
2,92230008,192 Avenue Louis Roche,Gennevilliers,92230,Hauts-de-Seine,Île-de-France,92,4893270,230440,2.149,2.099,2.149,1.999,NaN,NaN,"[Gazole, SP95, E10, SP98]",Non,2026-07-20 13:00:01.911887
3,34150003,Route de Pézenas,Gignac,34150,Hérault,Occitanie,34,4365100,354700,2.039,NaN,2.010,1.908,0.809,NaN,"[Gazole, E85, E10, SP98]",Oui,2026-07-20 13:00:01.911887
4,38230003,Rue des Ardennes,Tignieu-Jameyzieu,38230,Isère,Auvergne-Rhône-Alpes,38,4574500,518300,2.028,1.929,1.975,1.930,0.768,0.998,"[Gazole, SP95, E85, GPLc, E10, SP98]",Oui,2026-07-20 13:00:01.911887


## Résumé de l'exploration

- Les données sont chargées depuis S3 et organisées dans un DataFrame pandas.
- Les prix par carburant ont été analysés.
- Les valeurs extrêmes ne seront pas supprimées automatiquement, car elles peuvent être réelles selon la station, le carburant ou la période.
- En dbt, on gardera les données utiles et on ne supprimera que les valeurs clairement erronées.
- Les transformations dbt feront le nettoyage, le typage et la préparation des données pour les modèles silver et gold.

## Suite : logique de transformation pour dbt

Cette cellule montre la logique de nettoyage à reproduire dans les modèles dbt.
On garde les prix plausibles même s'ils semblent élevés, et on supprime uniquement les valeurs clairement invalides.

In [48]:
df_silver = df[cols_utiles].copy()

# Corriger les types
df_silver["latitude"] = pd.to_numeric(
    df_silver["latitude"], errors="coerce")
df_silver["longitude"] = pd.to_numeric(
    df_silver["longitude"], errors="coerce")
df_silver["cp"] = df_silver["cp"].astype(str).str.zfill(5)

# Supprimer uniquement les erreurs évidentes
cols_prix = ["gazole_prix", "sp95_prix", "sp98_prix",
             "e10_prix", "e85_prix", "gplc_prix"]

for col in cols_prix:
    df_silver[col] = df_silver[col].where(
        df_silver[col].between(0.50, 5.00)
    )

# Ajouter date de collecte
df_silver["date_collecte"] = datetime.now()

# Supprimer stations sans aucun prix
df_silver = df_silver.dropna(subset=cols_prix, how="all")

print(f"✅ Stations après nettoyage : {len(df_silver)}")
print(f"\n❌ Valeurs manquantes après nettoyage :")
print(df_silver[cols_prix].isnull().sum())
df_silver.head()

✅ Stations après nettoyage : 9669

❌ Valeurs manquantes après nettoyage :
gazole_prix      83
sp95_prix      6634
sp98_prix      1946
e10_prix       2172
e85_prix       5779
gplc_prix      8210
dtype: int64


,id,adresse,ville,cp,departement,region,code_departement,latitude,longitude,gazole_prix,sp95_prix,sp98_prix,e10_prix,e85_prix,gplc_prix,carburants_disponibles,horaires_automate_24_24,date_collecte
0,25115001,Z.A. les Salines,Pouilley-les-Vignes,25115,Doubs,Bourgogne-Franche-Comté,25,4.725741e+06,592408.728857,2.099,2.080,2.110,NaN,0.799,0.999,"[Gazole, SP95, E85, GPLc, SP98]",Oui,2026-07-21 14:46:16.719024
1,56580001,16 rue chateaubriand,Bréhan,56580,Morbihan,Bretagne,56,4.806100e+06,-269000.000000,2.149,2.099,2.129,NaN,NaN,NaN,"[Gazole, SP95, SP98]",Oui,2026-07-21 14:46:16.719024
2,92230008,192 Avenue Louis Roche,Gennevilliers,92230,Hauts-de-Seine,Île-de-France,92,4.893270e+06,230440.000000,2.149,2.099,2.149,1.999,NaN,NaN,"[Gazole, SP95, E10, SP98]",Non,2026-07-21 14:46:16.719024
3,34150003,Route de Pézenas,Gignac,34150,Hérault,Occitanie,34,4.365100e+06,354700.000000,2.039,NaN,2.010,1.908,0.809,NaN,"[Gazole, E85, E10, SP98]",Oui,2026-07-21 14:46:16.719024
4,38230003,Rue des Ardennes,Tignieu-Jameyzieu,38230,Isère,Auvergne-Rhône-Alpes,38,4.574500e+06,518300.000000,2.028,1.929,1.975,1.930,0.768,0.998,"[Gazole, SP95, E85, GPLc, E10, SP98]",Oui,2026-07-21 14:46:16.719024


## Coordonnées : préparation pour la géolocalisation

Pour la suite du projet, les coordonnées doivent être nettoyées et standardisées afin d'être utilisables pour la cartographie et les analyses géographiques.

In [49]:
from pyproj import Transformer

# 1) Conversion des coordonnées en GPS
transformer = Transformer.from_crs("EPSG:32631", "EPSG:4326", always_xy=True)

# Recréer la base à partir des colonnes utiles
cols_utiles = [
    "id", "adresse", "ville", "cp",
    "departement", "region", "code_departement",
    "latitude", "longitude",
    "gazole_prix", "sp95_prix", "sp98_prix",
    "e10_prix", "e85_prix", "gplc_prix",
    "carburants_disponibles", "horaires_automate_24_24"
]

df_silver = df[cols_utiles].copy()
df_silver["latitude"] = pd.to_numeric(df_silver["latitude"], errors="coerce")
df_silver["longitude"] = pd.to_numeric(df_silver["longitude"], errors="coerce")

def convertir_coords(row):
    try:
        lon, lat = transformer.transform(row["longitude"], row["latitude"])
        return pd.Series({"lat_gps": lat, "lon_gps": lon})
    except Exception:
        return pd.Series({"lat_gps": None, "lon_gps": None})

print("🔄 Conversion des coordonnées...")
coords = df_silver.apply(convertir_coords, axis=1)
df_silver["lat_gps"] = coords["lat_gps"]
df_silver["lon_gps"] = coords["lon_gps"]

# 2) Ensuite, appliquer le filtre sur les prix
cols_prix = ["gazole_prix", "sp95_prix", "sp98_prix", "e10_prix", "e85_prix", "gplc_prix"]
for col in cols_prix:
    df_silver[col] = pd.to_numeric(df_silver[col], errors="coerce")

# Garder uniquement les lignes avec des coordonnées converties et au moins un prix valide
mask_valides = (
    df_silver["lat_gps"].notna() &
    df_silver["lon_gps"].notna() &
    df_silver[cols_prix].notna().any(axis=1)
)
df_silver = df_silver[mask_valides]

print(f"✅ Stations avec coordonnées valides et prix présents : {len(df_silver)}")
print(df_silver[["ville", "lat_gps", "lon_gps"]].head())

🔄 Conversion des coordonnées...
✅ Stations avec coordonnées valides et prix présents : 9669
                 ville    lat_gps   lon_gps
0  Pouilley-les-Vignes  42.678598  4.127866
1               Bréhan  43.017007 -6.434356
2        Gennevilliers  44.142906 -0.370222
3               Gignac  39.423194  1.311957
4    Tignieu-Jameyzieu  41.321764  3.218661
